In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "utils" / "paths.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook inside the project directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import RAW, stage_path
from utils.runs import require_stage, stage_run
from loguru import logger
import csv
import os
import re
import shutil
import subprocess
from tqdm.auto import tqdm


In [ ]:
PREPARATION = require_stage("06_preparation")
LOCAL_VINA = PROJECT_ROOT / "tools" / "bin" / "vina.exe"
VINA_PATH = str(LOCAL_VINA) if LOCAL_VINA.is_file() else shutil.which("vina")
if VINA_PATH is None:
    raise FileNotFoundError("AutoDock Vina not found in tools/bin or PATH")
CENTER = (28.249, 0.618, 18.996)
SIZE = (22.904, 17.276, 22.017)
EXHAUSTIVENESS = 16
CPU = max(1, (os.cpu_count() or 1) - 2)
SEED = 42


In [ ]:
with stage_run("07_docking", [PREPARATION], {"center": CENTER, "size": SIZE, "exhaustiveness": EXHAUSTIVENESS, "cpu": CPU, "seed": SEED, "vina": VINA_PATH}) as output_dir:
    poses = output_dir / "poses"
    vina_logs = output_dir / "vina_logs"
    poses.mkdir()
    vina_logs.mkdir()
    with (PREPARATION / "preparation_results.csv").open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream)
        columns = list(reader.fieldnames or [])
        rows = [row for row in reader if row["PreparationStatus"] == "OK"]
    results = []
    for row in tqdm(rows, desc="Docking ligands", unit="mol"):
        ligand_id = row["LigandID"]
        command = [VINA_PATH, "--receptor", str(PREPARATION / "receptor.pdbqt"),
                   "--ligand", str(PREPARATION / "ligands" / f"{ligand_id}.pdbqt"),
                   "--out", str(poses / f"{ligand_id}_out.pdbqt"),
                   "--exhaustiveness", str(EXHAUSTIVENESS), "--cpu", str(CPU), "--seed", str(SEED)]
        for axis, center, size in zip("xyz", CENTER, SIZE):
            command.extend([f"--center_{axis}", str(center), f"--size_{axis}", str(size)])
        affinity, status, error = "", "OK", ""
        try:
            result = subprocess.run(command, capture_output=True, text=True)
            (vina_logs / f"{ligand_id}.log").write_text(
                f"Command: {command!r}\nReturn code: {result.returncode}\nSTDOUT:\n{result.stdout}\nSTDERR:\n{result.stderr}", encoding="utf-8")
            if result.returncode != 0:
                raise RuntimeError(f"Vina exited with code {result.returncode}: {result.stderr}")
            match = re.search(r"^\s*1\s+(-?\d+(?:\.\d+)?)\s", result.stdout, re.MULTILINE)
            if not match:
                raise RuntimeError("Vina affinity not found in output")
            affinity = float(match.group(1))
            if not (poses / f"{ligand_id}_out.pdbqt").is_file():
                raise RuntimeError("Vina pose file missing")
        except Exception as exc:
            status, error = "ERROR", str(exc)
            logger.exception("Docking failed: {}", ligand_id)
        results.append(dict(row, Affinity=affinity, DockingStatus=status, DockingError=error))
    results.sort(key=lambda row: float(row["Affinity"]) if row["DockingStatus"] == "OK" else float("inf"))
    with (output_dir / "docking_scores.csv").open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns + ["Affinity", "DockingStatus", "DockingError"])
        writer.writeheader()
        writer.writerows(results)
    logger.info("Docking records: {} | Successful: {}", len(results), sum(r["DockingStatus"] == "OK" for r in results))
